<p align="center">
  <img src="https://raw.githubusercontent.com/paulmunozpauta/Hidrology_Course/main/Static/Imgs/UdeC_color_horizontal.jpg" width="500">
</p>
<p align="center"><b style="font-size:28px;">Facultad de Ingeniería Agrícola</b></p>
<p align="center"><b style="font-size:28px;">Curso de Hidrología</b></p>
<hr>
<p align="center"><b style="font-size:28px;">Contacto</b></p>
<p align="center">
  paulmunoz@udec.cl<br>
  https://paulmunoz.com
</p>

# Precipitación

## Objetivo

En este notebook aprenderemos a **procesar, analizar y evaluar datos pluviométricos** de una estación real en Chile, utilizando herramientas de Python orientadas al análisis hidrológico.

Al finalizar este notebook, el estudiante será capaz de:

- Leer y preprocesar series de tiempo de precipitación en formato DGA.
- Visualizar registros diarios y acumulados.
- Aplicar métodos de **relleno de datos faltantes** (promedio simple y razón normal).
- Evaluar la **homogeneidad** de una serie mediante la curva de doble masa.
- Calcular e interpretar la **intensidad de precipitación**.

---


**Fuente de datos:** En Chile, la Dirección General de Aguas (DGA) y la Dirección Meteorológica de Chile (DMC) mantienen redes de estaciones pluviométricas. Los datos están disponibles en: https://explorador.cr2.cl/


# 1. Configuración del entorno

## 1.1 Clonar el repositorio desde GitHub

Para acceder a los datos y scripts del curso, clonaremos el repositorio desde GitHub. Esto descarga todos los archivos necesarios en el entorno de Google Colab.

> ⚠️ **Nota:** Esta celda solo debe ejecutarse **una vez** por sesión. Si el repositorio ya fue clonado, puedes omitir este paso.

In [ ]:
# Clona el repositorio del curso desde GitHub en el entorno de Google Colab.
!git clone -- https://github.com/paulmunozpauta/Hidrology_Course

Después de ejecutar la celda anterior, se crea una carpeta llamada **`Hidrology_Course`** en el entorno de trabajo. Esta carpeta contiene todos los archivos del curso, incluyendo los datos pluviométricos que usaremos a continuación.

## 1.2 Ingresar a la carpeta del repositorio

Cambiamos el directorio de trabajo para que Python pueda encontrar los archivos de datos mediante rutas relativas.

# Entrar a la carpeta del repositorio





In [ ]:
# Cambia el directorio de trabajo actual a la carpeta del repositorio clonado.
# El comando mágico %cd de Jupyter/Colab cambia el directorio de forma persistente
%cd Hidrology_Course
# Muestra los archivos disponibles en la carpeta actual para verificar
# que el repositorio se clonó correctamente.
!ls


# 2. Instalación e importación de librerías

Las siguientes librerías son fundamentales para el análisis hidrológico en Python:

| Librería | Uso principal en este notebook |
|----------|--------------------------------|
| `pandas` | Manejo de series de tiempo y DataFrames |
| `numpy` | Operaciones numéricas y manejo de valores nulos |
| `matplotlib` | Visualización de datos (gráficos y figuras) |
| `os` | Interacción con el sistema de archivos |



Librerías principales:
pandas
numpy
matplotlib

In [ ]:
# Instala (o actualiza) la librería pandas desde PyPI.
!pip install -q pandas

In [ ]:
# os: permite interactuar con rutas y directorios del sistema operativo.
import os

# numpy: librería de computación numérica. Se usa para operaciones vectoriales
# y para trabajar con valores nulos (np.nan).
import numpy as np

# matplotlib.pyplot: módulo de visualización. Permite crear gráficos de líneas,
# barras, dispersión, etc.
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches


# pandas: librería principal para análisis de datos tabulares y series de tiempo.
# Se usará para leer los archivos Excel y manipular los datos pluviométricos.
import pandas as pd
import random
import math
import struct


print("✅ Librerías importadas correctamente.")


# 3. Lectura de datos hidrometeorológicos

## Estación Puerto Montt – DGA (código 10425001)

Trabajaremos con datos de la estación **Puerto Montt**, ubicada en la Región de Los Lagos. Esta estación pertenece a la red de la Dirección General de Aguas (DGA) de Chile.

**¿Por qué Puerto Montt?**

Puerto Montt es una de las ciudades más lluviosas de Chile, con precipitaciones anuales que superan los 1.800 mm. Su régimen pluviométrico es del tipo **oceánico templado**, con lluvias distribuidas a lo largo de todo el año y un leve máximo invernal.

**Formato de los datos DGA:**

Los archivos descargados del explorador del CR2 (https://explorador.cr2.cl/) suelen tener columnas separadas para año (`agno`), mes (`mes`), día (`dia`) y valor de precipitación diaria (`valor`), en lugar de una columna de fecha completa. Este formato requiere un paso de preprocesamiento que veremos a continuación.

In [ ]:
# Define la ruta relativa al archivo de datos de la estación Puerto Montt.
# Al haber cambiado el directorio con %cd, la ruta parte desde la raíz del repositorio.
archivo = "Data/Precipitación/PuertoMontt_DGA_10425001.xlsx"

# Lee el archivo Excel y carga los datos en un DataFrame de pandas.
# Un DataFrame es una tabla bidimensional con filas y columnas etiquetadas.
PMontt = pd.read_excel(archivo)

# Muestra las primeras filas del DataFrame para verificar la estructura de los datos.
print(f"Dimensiones del dataset: {PMontt.shape[0]} filas × {PMontt.shape[1]} columnas\n")
PMontt.head(10)


# 4. Preprocesamiento de datos

## ¿Por qué es necesario el preprocesamiento?

Los datos crudos de la DGA presentan varias situaciones que deben resolverse antes de cualquier análisis:

1. **Columnas separadas de año, mes y día:** Deben unirse para formar una fecha válida.
2. **Nombres de columnas inconsistentes:** Pueden tener espacios, acentos o variaciones    ortográficas según la versión del archivo.
3. **Valores no numéricos:** Algunos registros pueden contener texto (e.g., códigos de error).    Se deben convertir a numérico, forzando los no válidos a `NaN`.
4. **Fechas inválidas:** Por ejemplo, el 31 de febrero, que no existe. Deben eliminarse.
5. **Ordenamiento temporal:** Los datos deben estar ordenados cronológicamente.

El preprocesamiento es un paso crítico en hidrología: datos mal formateados pueden producir errores en cálculos estadísticos, análisis de frecuencia y diseño hidrológico.

Los datos diarios de esta estación están separados en columnas por "agno", "mes", "día" y "valor".

Vamos a generar una serie (dataframe) ordenada cronológicamente a través de la fecha disponible

In [ ]:
# limpiar nombres de columnas
PMontt.columns = PMontt.columns.str.strip().str.lower()

# renombrar por seguridad si vienen con espacios o acentos raros
PMontt = PMontt.rename(columns={
    "agno": "anio",
    "año": "anio",
    "mes": "mes",
    "dia": "dia",
    "valor": "precipitacion_diaria_mm"
})

# convertir columnas a numéricas
for col in ["anio", "mes", "dia", "precipitacion_diaria_mm"]:
    PMontt[col] = pd.to_numeric(PMontt[col], errors="coerce")

# eliminar filas con datos faltantes en columnas clave
PMontt = PMontt.dropna(subset=["anio", "mes", "dia", "precipitacion_diaria_mm"])

# convertir año, mes y día a enteros
PMontt["anio"] = PMontt["anio"].astype(int)
PMontt["mes"] = PMontt["mes"].astype(int)
PMontt["dia"] = PMontt["dia"].astype(int)

# crear columna fecha
PMontt["fecha"] = pd.to_datetime(
    dict(year=PMontt["anio"], month=PMontt["mes"], day=PMontt["dia"]),
    errors="coerce"
)

# eliminar fechas inválidas
PMontt = PMontt.dropna(subset=["fecha"])

# dejar solo la serie temporal final
PMontt = PMontt[["fecha", "precipitacion_diaria_mm"]].copy()

# ordenar por fecha
PMontt = PMontt.sort_values("fecha").reset_index(drop=True)

# opcional: usar fecha como índice
PMontt = PMontt.set_index("fecha")

# mostrar la serie
PMontt


# 5. Visualización de la serie de precipitación diaria

Antes de realizar cualquier análisis, es fundamental **visualizar los datos** para:
- Detectar tendencias generales y estacionalidad.
- Identificar posibles errores o datos atípicos (*outliers*).
- Comprender el régimen pluviométrico de la estación.

>  **¿Por qué no graficamos como barras toda la serie?**
>
> La precipitación diaria se representa conceptualmente como barras, ya que cada valor corresponde a una acumulación en un intervalo de tiempo. Sin embargo, graficar decenas de miles de barras tiene un costo computacional elevado y la visualización pierde claridad. Para la serie completa usaremos líneas; las barras las reservamos para períodos más cortos.


In [ ]:
plt.figure(figsize=(12,5))
PMontt.plot()#no lo hacemos como barras por la carga computacional
plt.xlabel("Fecha")
plt.ylabel("Precipitación (mm)")
plt.grid(axis="y")
plt.show()

# 6. Relleno de datos faltantes

## ¿Por qué faltan datos en estaciones pluviométricas?

En la práctica, las series de precipitación raramente son completas. Los registros pueden faltar por fallas del instrumento, problemas de comunicación, errores del observador, o simplemente porque la estación no operaba en ese período.

**Impacto de los datos faltantes:**

Los datos faltantes afectan directamente los cálculos de:
- Precipitación media anual (módulo pluviométrico)
- Balances hídricos
- Análisis de frecuencia de eventos extremos

## Criterio para elegir el método de relleno

La elección del método depende de qué tan diferente es el régimen pluviométrico de la estación con datos faltantes ($P_x$) respecto a las estaciones vecinas usadas como referencia.

El criterio se basa en el **módulo pluviométrico** ($M$): la precipitación media anual histórica de cada estación.

| Condición | Método recomendado |
|-----------|--------------------|
| Diferencia entre módulos < 10% | Método 1: Promedio aritmético simple |
| Diferencia entre módulos ≥ 10% | Método 2: Razón normal ponderada |

## Visualización del año 2022 (antes de introducir datos faltantes)

Para ilustrar el proceso, primero visualizamos el año 2022 completo en forma de barras diarias y luego como precipitación acumulada.

In [ ]:

# ── Gráfico 1: Precipitación diaria de 2022 en barras ────────────────────────
# Seleccionamos solo los datos del año 2022 usando indexación por etiqueta de fecha.
data_2022 = PMontt.loc["2022"]

fig, ax = plt.subplots(figsize=(15, 5))
data_2022.plot(kind="bar", ax=ax, color="steelblue", width=1.0, legend=False)

# Configuramos las etiquetas del eje X para mostrar solo el nombre del mes
# (en lugar de cada día, que sería ilegible).
dates = data_2022.index
month_positions = [i for i, d in enumerate(dates) if d.day == 1]
month_labels    = [d.strftime("%b") for d in dates if d.day == 1]
ax.set_xticks(month_positions)
ax.set_xticklabels(month_labels, rotation=0)

plt.xlabel("Mes", fontsize=11)
plt.ylabel("Precipitación diaria (mm)", fontsize=11)
plt.title("Precipitación diaria – Puerto Montt (2022)", fontsize=13)
plt.grid(axis="y", linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

In [ ]:
# ── Gráfico 2: Precipitación acumulada de 2022 ───────────────────────────────
# .cumsum() calcula la suma acumulada: cada punto representa la precipitación
# total desde el 1 de enero hasta esa fecha.
# Esta representación es útil para detectar períodos secos o húmedos inusualmente largos.
fig, ax = plt.subplots(figsize=(14, 4))
data_2022.cumsum().plot(ax=ax, color="darkblue", linewidth=1.5, legend=False)

plt.xlabel("Fecha", fontsize=11)
plt.ylabel("Precipitación acumulada (mm)", fontsize=11)
plt.title("Precipitación acumulada – Puerto Montt (2022)", fontsize=13)
plt.grid(linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

## 6.1 Simulación de datos faltantes

Para demostrar los métodos de relleno, **introduciremos artificialmente** datos faltantes en el período del 1 al 15 de julio de 2022. Esto nos permitirá comparar los valores estimados por cada método contra los valores originales.

In [ ]:
# Guardamos una copia de la serie original ANTES de eliminar datos.
# Esta copia nos servirá posteriormente para evaluar la calidad del relleno
# comparando los valores estimados con los valores reales.
PMontt_original = PMontt.copy()

# Reemplazamos los valores del 1 al 15 de julio de 2022 con NaN (Not a Number),
# que es la forma estándar de representar datos faltantes en pandas/numpy.
PMontt.loc["2022-07-01":"2022-07-15"] = np.nan

print("Datos eliminados en el período 2022-07-01 a 2022-07-15:")
print(f"  Registros con NaN: {PMontt.loc['2022-07-01':'2022-07-15'].isna().sum().values[0]}")

In [ ]:
# Verificamos visualmente el efecto de los datos faltantes en la curva acumulada.
# La curva debería "aplanarse" (dejar de crecer) durante el período sin datos.
fig, ax = plt.subplots(figsize=(14, 4))
PMontt.loc["2022"].cumsum().plot(ax=ax, color="darkblue", linewidth=1.5, legend=False)

plt.xlabel("Fecha", fontsize=11)
plt.ylabel("Precipitación acumulada (mm)", fontsize=11)
plt.title("Precipitación acumulada 2022 – CON DATOS FALTANTES (jul 1–15)", fontsize=13)
plt.grid(linestyle="--", alpha=0.6)

# Marcamos el período faltante con una banda gris para visualizarlo claramente.
ax.axvspan(pd.Timestamp("2022-07-01"), pd.Timestamp("2022-07-15"),
           alpha=0.2, color="red", label="Período faltante")
ax.legend()
plt.tight_layout()
plt.show()

## 6.2 Selección de estaciones vecinas

Para aplicar los métodos de relleno, necesitamos al menos tres estaciones cercanas con datos simultáneos y alta correlación con la estación en análisis.

Las estaciones disponibles se pueden verificar en: https://explorador.cr2.cl/

<p align="center">
  <img src="https://raw.githubusercontent.com/paulmunozpauta/Hidrology_Course/main/Static/Imgs/PMontt_estaciones_cercanas.png" width="800">
</p>

Las estaciones candidatas son:

| Código | Nombre | Fuente |
|--------|--------|--------|
| 10430005 | Lago Chapo | DGA |
| 10523001 | Puelo | DGA |
| 410005 | Tepual (Aeropuerto) | DMC |
| 10410015 | Cascada | DGA |

> **¿Cómo elegir las estaciones?** Se recomienda seleccionar estaciones que estén:
> - Dentro de la misma cuenca o región climática.
> - Preferiblemente a menos de 50–100 km de distancia.
> - Con alta correlación estadística (p.ej., r > 0.7) con la estación objetivo.

### Función de preprocesamiento reutilizable

Para cargar las cuatro estaciones vecinas, encapsulamos el proceso de preprocesamiento en una **función** reutilizable. Esto aplica el principio de programación **DRY** (*Don't Repeat Yourself*): evitamos duplicar código y reducimos el riesgo de errores.

In [ ]:
def preparar_serie_precipitacion(df):
    """
    Preprocesa un DataFrame con datos pluviométricos en formato DGA.

    El formato DGA presenta las columnas de año (agno), mes, día y valor
    separadas. Esta función las unifica en una serie de tiempo indexada por fecha.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame con columnas: agno/año, mes, dia/día, valor/precipitacion.

    Returns
    -------
    pd.DataFrame
        Serie temporal con índice de tipo DatetimeIndex y columna
        "precipitacion_diaria_mm", ordenada cronológicamente.
    """
    df = df.copy()  # Evitamos modificar el DataFrame original (buena práctica)

    # Normalizar nombres de columnas
    df.columns = df.columns.str.strip().str.lower()

    # Mapa de posibles nombres de columnas en archivos DGA/DMC
    df = df.rename(columns={
        "agno": "anio", "año": "anio", "ano": "anio",
        "mes": "mes",
        "dia": "dia", "día": "dia",
        "valor": "precipitacion_diaria_mm",
        "precipitacion": "precipitacion_diaria_mm",
        "precipitación": "precipitacion_diaria_mm",
        "precipitacion_diaria": "precipitacion_diaria_mm"
    })

    # Convertir a numérico y limpiar nulos
    for col in ["anio", "mes", "dia", "precipitacion_diaria_mm"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df = df.dropna(subset=["anio", "mes", "dia", "precipitacion_diaria_mm"])

    # Convertir a enteros (requerido por pd.to_datetime)
    df["anio"] = df["anio"].astype(int)
    df["mes"]  = df["mes"].astype(int)
    df["dia"]  = df["dia"].astype(int)

    # Construir columna de fecha y eliminar fechas inválidas
    df["fecha"] = pd.to_datetime(
        dict(year=df["anio"], month=df["mes"], day=df["dia"]),
        errors="coerce"
    )
    df = df.dropna(subset=["fecha"])

    # Retornar serie ordenada con fecha como índice
    df = df[["fecha", "precipitacion_diaria_mm"]].copy()
    df = df.sort_values("fecha").reset_index(drop=True)
    return df.set_index("fecha")

print("✅ Función preparar_serie_precipitacion() definida.")

In [ ]:
# Cargamos los archivos Excel de las cuatro estaciones vecinas.
# pd.read_excel() lee el archivo directamente en un DataFrame.
LagoChapo_raw = pd.read_excel("Data/Precipitación/LagoChapo_Lm_DGA_10430005.xlsx")
Puelo_raw     = pd.read_excel("Data/Precipitación/Puelo_DGA_10523001.xlsx")
Tepual_raw    = pd.read_excel("Data/Precipitación/Tepual_DMC_410005.xlsx")
Cascada_raw   = pd.read_excel("Data/Precipitación/Cascada_DGA_10410015.xlsx")

# Aplicamos la función de preprocesamiento a cada estación.
LagoChapo = preparar_serie_precipitacion(LagoChapo_raw)
Puelo     = preparar_serie_precipitacion(Puelo_raw)
Tepual    = preparar_serie_precipitacion(Tepual_raw)
Cascada   = preparar_serie_precipitacion(Cascada_raw)

print("✅ Estaciones cargadas y preprocesadas:")
for nombre, est in [("Lago Chapo", LagoChapo), ("Puelo", Puelo),
                     ("Tepual", Tepual), ("Cascada", Cascada)]:
    print(f"  {nombre}: {len(est)} registros ({est.index.min().date()} → {est.index.max().date()})")

## 6.3 Análisis de correlación

Para seleccionar las **tres mejores estaciones** de relleno, calculamos el **coeficiente de correlación de Pearson** entre la precipitación diaria de Puerto Montt y cada estación vecina.

$$r = \frac{\sum (P_x - \bar{P}_x)(P_i - \bar{P}_i)}{\sqrt{\sum (P_x - \bar{P}_x)^2 \sum (P_i - \bar{P}_i)^2}}$$

El coeficiente $r$ varía entre -1 y 1:
- $r$ cercano a **1**: correlación positiva fuerte (llueve más en ambas al mismo tiempo).
- $r$ cercano a **0**: sin correlación.
- $r$ cercano a **-1**: correlación negativa (cuando llueve en una, no llueve en la otra).

In [ ]:
# Concatenamos todas las series en un único DataFrame alineado por fecha.
# pd.concat con axis=1 une columnas (series) compartiendo el índice de fecha.
# Las fechas sin dato en alguna estación quedarán como NaN automáticamente.
df_corr = pd.concat(
    [
        PMontt["precipitacion_diaria_mm"].rename("PuertoMontt"),
        LagoChapo["precipitacion_diaria_mm"].rename("LagoChapo"),
        Puelo["precipitacion_diaria_mm"].rename("Puelo"),
        Tepual["precipitacion_diaria_mm"].rename("Tepual"),
        Cascada["precipitacion_diaria_mm"].rename("Cascada")
    ],
    axis=1
)

# .corr() calcula la matriz de correlación de Pearson entre todas las columnas.
# Seleccionamos solo la columna de Puerto Montt y ordenamos de mayor a menor.
correlaciones = df_corr.corr()["PuertoMontt"].sort_values(ascending=False)

print("Correlación de Pearson con Puerto Montt:")
print(correlaciones.to_string())
print("\n→ Se seleccionan las 3 estaciones con mayor correlación (excluyendo la propia).")


## 6.4 Método 1: Promedio aritmético simple

**Condición de aplicación:** La precipitación media anual de las estaciones vecinas difiere en **menos del 10%** respecto a la estación con datos faltantes.

**Fórmula:**

$$P_x = \frac{P_a + P_b + P_c}{3}$$

Donde $P_a$, $P_b$ y $P_c$ son las precipitaciones registradas en las tres estaciones vecinas para el mismo día que falta en la estación $P_x$.

**Interpretación:** Este método asume que todas las estaciones contribuyen igualmente al estimado. Es simple y efectivo cuando las estaciones presentan condiciones climáticas similares.

In [ ]:
# Unimos todas las series en un DataFrame consolidado.
# Incluimos tanto la serie con datos faltantes como la original (para comparar).
df_todas_estaciones = pd.concat(
    [
        PMontt["precipitacion_diaria_mm"].rename("PuertoMontt"),
        PMontt_original["precipitacion_diaria_mm"].rename("PuertoMontt_original"),
        Tepual["precipitacion_diaria_mm"].rename("Tepual"),
        Puelo["precipitacion_diaria_mm"].rename("Puelo"),
        LagoChapo["precipitacion_diaria_mm"].rename("LagoChapo"),
    ],
    axis=1
)

# Visualizamos el período alrededor de los datos faltantes para verificar la alineación.
print("Datos en torno al período faltante (2022-06-28 a 2022-07-20):")
df_todas_estaciones.loc["2022-06-28":"2022-07-20"]

In [ ]:
# ── Aplicación del Método 1: Promedio aritmético simple ──────────────────────

# Inicializamos la columna de relleno con NaN.
# Solo rellenaremos el período faltante; el resto permanece sin valores estimados.
df_todas_estaciones["Relleno_M1"] = np.nan

# Definimos el slice (período) a rellenar.
periodo_relleno = slice("2022-07-01", "2022-07-15")

# Calculamos el promedio diario de las tres estaciones vecinas seleccionadas.
# .mean(axis=1) calcula el promedio fila a fila (es decir, día a día).
df_todas_estaciones.loc[periodo_relleno, "Relleno_M1"] = (
    df_todas_estaciones.loc[periodo_relleno, ["Tepual", "Puelo", "LagoChapo"]]
    .mean(axis=1)
)

# Comparamos los valores estimados con los valores originales.
print("Comparación Método 1 – Período 1-15 julio 2022:")
df_todas_estaciones.loc[
    periodo_relleno,
    ["PuertoMontt_original", "PuertoMontt", "Relleno_M1", "Tepual", "Puelo", "LagoChapo"]
]


## 6.5 Método 2: Razón normal ponderada

**Condición de aplicación:** Cuando la precipitación media anual de las estaciones vecinas difiere en **más del 10%** respecto a la estación objetivo. Es el método más robusto.

**Fórmula:**

$$P_x = M_x \cdot \frac{1}{3}\left(\frac{P_a}{M_a} + \frac{P_b}{M_b} + \frac{P_c}{M_c}\right)$$

Donde:
- $M_x, M_a, M_b, M_c$ = precipitación **media anual histórica** de cada estación (módulo pluviométrico).
- $P_a, P_b, P_c$ = precipitación observada en las estaciones vecinas para el día a rellenar.

**Interpretación:** Este método **normaliza** los valores de precipitación por el módulo pluviométrico antes de promediar. Así, estaciones más lluviosas no dominan artificialmente el estimado. Es equivalente a expresar cada precipitación como fracción de la media anual antes de combinar.

In [ ]:
# ── Método 2: Razón normal ponderada ─────────────────────────────────────────

# PASO 1: Calcular la precipitación anual total por estación.
# resample("Y").sum() agrupa los datos por año y suma los valores.
# min_count=300 exige al menos 300 días con dato para considerar el año válido
# (evita años incompletos que subestimarían el módulo).
precipitacion_anual = df_todas_estaciones[
    ["PuertoMontt_original", "Tepual", "Puelo", "LagoChapo"]
].resample("Y").sum(min_count=300)

# PASO 2: Calcular el módulo pluviométrico (media de años válidos).
# Este es el valor histórico de referencia para normalizar.
modulos = precipitacion_anual.mean()

print("Módulos pluviométricos (precipitación media anual, mm):")
for est, val in modulos.items():
    print(f"  {est}: {val:.1f} mm/año")

# Extraemos los módulos de cada estación
Mx = modulos["PuertoMontt_original"]  # Módulo de la estación objetivo
Ma = modulos["Tepual"]
Mb = modulos["Puelo"]
Mc = modulos["LagoChapo"]

# Verificamos si los módulos difieren en más del 10% (criterio de selección del método)
dif_max = max(abs(Ma - Mx), abs(Mb - Mx), abs(Mc - Mx)) / Mx * 100
print(f"\nDiferencia máxima entre módulos: {dif_max:.1f}%")
if dif_max >= 10:
    print("→ Se recomienda usar el Método 2 (razón normal).")
else:
    print("→ El Método 1 (promedio simple) es suficiente.")

Compracion de curvas acumuladas

In [ ]:
# PASO 3: Aplicar la fórmula de razón normal al período faltante.

# Inicializamos la columna de relleno por razón normal.
df_todas_estaciones["Relleno_M2"] = np.nan

# Aplicamos la fórmula: Px = Mx * promedio(Pi/Mi)
# Cada término Pi/Mi expresa la lluvia del día como fracción de la media anual.
# Al multiplicar por Mx, "traducimos" ese ratio a la escala de Puerto Montt.
df_todas_estaciones.loc[periodo_relleno, "Relleno_M2"] = Mx * (
    (
        df_todas_estaciones.loc[periodo_relleno, "Tepual"]    / Ma
      + df_todas_estaciones.loc[periodo_relleno, "Puelo"]     / Mb
      + df_todas_estaciones.loc[periodo_relleno, "LagoChapo"] / Mc
    ) / 3
)

# Comparación final de ambos métodos con el valor original
print("Comparación de métodos de relleno – 1-15 julio 2022:")
df_todas_estaciones.loc[
    periodo_relleno,
    ["PuertoMontt_original", "Relleno_M1", "Relleno_M2"]
]

## 6.6 Comparación de métodos mediante curvas acumuladas

La **curva de precipitación acumulada** es una herramienta útil para comparar visualmente la calidad del relleno. Si el método es bueno, la curva estimada debería aproximarse a la curva original en el período de relleno.

In [ ]:
# Construimos un DataFrame solo con el período de relleno.
# Comparamos: valor original vs. estimaciones de Método 1 y Método 2.
df_comparacion = df_todas_estaciones.loc[
    "2022-07-01":"2022-07-15",
    ["PuertoMontt_original", "Relleno_M1", "Relleno_M2"]
].cumsum()

# Renombramos para la leyenda del gráfico
df_comparacion.columns = ["Original", "Método 1 (promedio simple)", "Método 2 (razón normal)"]

fig, ax = plt.subplots(figsize=(12, 5))
df_comparacion.plot(ax=ax, linewidth=2)

plt.xlabel("Fecha", fontsize=11)
plt.ylabel("Precipitación acumulada (mm)", fontsize=11)
plt.title("Comparación de métodos de relleno – Puerto Montt (1–15 julio 2022)", fontsize=13)
plt.legend(fontsize=10)
plt.grid(linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

# Evaluación cuantitativa: diferencia acumulada total respecto al original
total_orig = PMontt_original.loc["2022-07-01":"2022-07-15", "precipitacion_diaria_mm"].sum()
total_m1   = df_todas_estaciones.loc["2022-07-01":"2022-07-15", "Relleno_M1"].sum()
total_m2   = df_todas_estaciones.loc["2022-07-01":"2022-07-15", "Relleno_M2"].sum()

print(f"Precipitación total original:       {total_orig:.1f} mm")
print(f"Estimado Método 1 (promedio):       {total_m1:.1f} mm  (error: {abs(total_m1-total_orig)/total_orig*100:.1f}%)")
print(f"Estimado Método 2 (razón normal):   {total_m2:.1f} mm  (error: {abs(total_m2-total_orig)/total_orig*100:.1f}%)")

# 7. Análisis de homogeneidad: Curva de doble masa

## ¿Qué es la homogeneidad de una serie?

Una serie pluviométrica es **homogénea** cuando los cambios registrados a lo largo del tiempo reflejan únicamente variaciones climatológicas reales, y no alteraciones en las condiciones de medición.

**Causas frecuentes de la no homogeneidad:**
- Cambio de instrumento (pluviómetro viejo → nuevo modelo)
- Reubicación de la estación (cambio de altitud, exposición al viento)
- Cambios en el entorno (urbanización, crecimiento de vegetación)
- Cambio de observador (diferentes criterios de lectura)

## Principio del método

La **curva de doble masa** compara la precipitación acumulada de la estación analizada ($\sum P_x$) contra la precipitación acumulada de una **serie patrón** ($\sum P_p$), construida como el promedio de estaciones vecinas representativas.

Si la estación es homogénea, la relación entre ambas acumuladas debería ser **lineal**:

$$\sum P_x \approx \alpha \cdot \sum P_p$$

**Interpretación:**

| Forma de la curva | Interpretación |
|-------------------|----------------|
| Línea recta continua | Serie homogénea  |
| Cambio de pendiente | Posible inhomogeneidad (revisar registros históricos)  |
| Salto brusco | Error puntual en uno o pocos registros  |

## Corrección de series no homogéneas

Si se detecta un quiebre claro, los datos del tramo afectado pueden corregirse:

$$P_c = P_m \cdot \frac{\alpha_1}{\alpha_i}$$

Donde $\alpha_1$ es la pendiente del período de referencia (generalmente el más reciente o el más confiable) y $\alpha_i$ es la pendiente del tramo con inconsistencia.

> ⚠️ **Precaución:** No se recomienda corregir si el quiebre dura menos de ~5 años o no tiene causa física o instrumental identificable.

In [ ]:
# ── Curva de doble masa: Puerto Montt vs. promedio de estaciones vecinas ──────

# PASO 1: Construir la serie patrón como promedio de las estaciones vecinas.
# Usamos .mean(axis=1) para calcular el promedio diario entre estaciones.
# Los días donde alguna estación no tiene dato son ignorados automáticamente
# (pandas omite NaN en el promedio).
df_todas_estaciones["Promedio_vecinas"] = df_todas_estaciones[
    ["Tepual", "Puelo", "LagoChapo"]
].mean(axis=1)

# PASO 2: Seleccionar fechas con datos completos en ambas series.
# Necesitamos días donde tanto Puerto Montt como el promedio de vecinas tengan dato.
df_doble_masa = df_todas_estaciones[
    ["PuertoMontt_original", "Promedio_vecinas"]
].dropna()

# PASO 3: Calcular los acumulados (suma progresiva en el tiempo).
# .cumsum() acumula los valores desde el primer registro hasta cada fecha.
df_doble_masa = df_doble_masa.copy()
df_doble_masa["PuertoMontt_acum"] = df_doble_masa["PuertoMontt_original"].cumsum()
df_doble_masa["Vecinas_acum"]     = df_doble_masa["Promedio_vecinas"].cumsum()

print(f"Pares de datos para la curva de doble masa: {len(df_doble_masa)}")

In [ ]:
# PASO 4: Graficar la curva de doble masa.

fig, ax = plt.subplots(figsize=(8, 7))

# Graficamos la acumulada de Puerto Montt (eje Y) vs. la acumulada del patrón (eje X).
# Si la serie es homogénea, los puntos deben seguir una línea recta.
ax.plot(
    df_doble_masa["Vecinas_acum"],
    df_doble_masa["PuertoMontt_acum"],
    color="steelblue", linewidth=1.5,
    label="Doble masa"
)

# Línea de referencia 1:1 (si ambas estaciones tuvieran el mismo módulo)
xmax = df_doble_masa["Vecinas_acum"].max()
ymax = df_doble_masa["PuertoMontt_acum"].max()
ax.plot([0, xmax], [0, ymax * (xmax / xmax)], "k--", alpha=0.4, label="Referencia lineal")

ax.set_xlabel("Precipitación acumulada – Promedio estaciones vecinas (mm)", fontsize=11)
ax.set_ylabel("Precipitación acumulada – Puerto Montt (mm)", fontsize=11)
ax.set_title("Curva de doble masa\nPuerto Montt vs. estaciones vecinas", fontsize=13)
ax.legend(fontsize=10)
ax.grid(linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

### Interpretación del resultado

La relación es **aproximadamente lineal** durante gran parte del registro histórico.

 Esto sugiere que la serie de Puerto Montt es **consistente** respecto a las estaciones vecinas:
- No se observan quiebres abruptos de pendiente que indiquen cambios instrumentales.
- No hay saltos verticales que sugieran errores groseros en la base de datos.
- La serie puede utilizarse con confianza para análisis estadísticos e hidrológicos.

### Ejemplos de curvas no homogéneas

A continuación se muestran ejemplos típicos donde la curva de doble masa evidencia cambios en las condiciones de medición:

<p align="center">
  <img src="https://raw.githubusercontent.com/paulmunozpauta/Hidrology_Course/main/Static/Imgs/Curva_doble_masa_1.png" width="500">
</p>
<p align="center"><i>Figura: Ejemplo de cambio de pendiente (posible cambio de instrumento o reubicación).</i></p>

<p align="center">
  <img src="https://raw.githubusercontent.com/paulmunozpauta/Hidrology_Course/main/Static/Imgs/Curva_doble_masa_2.png" width="500">
</p>
<p align="center"><i>Figura: Ejemplo de salto brusco (posible error puntual en un registro).</i></p>

# 8. Estimación espacial de la precipitación

## ¿Por qué no basta con medir en un punto?

Una estación pluviométrica registra la precipitación en un único punto del espacio.
Sin embargo, el diseño hidrológico requiere conocer la **precipitación media sobre
una cuenca o área** (en mm), que es la lámina equivalente de agua que cayó
uniformemente sobre toda la superficie.

Dado que la red de estaciones es finita y su distribución espacial suele ser
irregular (concentrada en zonas pobladas y accesibles, normalmente en cotas
bajas donde llueve menos), la estimación de la precipitación media espacial
introduce un error sistemático si no se aplica un método adecuado.

## Los tres métodos clásicos

La literatura hidrológica establece tres procedimientos de **precisión creciente**:

| Método | Complejidad | Considera posición espacial | Precisión |
|--------|-------------|----------------------------|-----------|
| Promedio aritmético simple | Baja | ✗ | Menor |
| Polígonos de Thiessen | Media |  (área de influencia) | Intermedia |
| Isoyetas | Alta |  (interpolación continua) | Mayor |



## 8.1 Promedio aritmético simple

Es la estimación más sencilla: el promedio no ponderado de todas las estaciones
dentro del área de estudio.

$$\bar{P} = \frac{\sum_{i=1}^{N} P_i}{N}$$

**Limitación:** Si las estaciones están concentradas en zonas bajas (donde llueve
menos), el promedio aritmético **subestimará sistemáticamente** la precipitación
real de la cuenca.



## 8.2 Polígonos de Thiessen

Cada estación recibe un peso proporcional al **área de influencia** que le corresponde
geométricamente. Esas áreas se construyen trazando las mediatrices (simetrales)
de los segmentos que unen estaciones adyacentes, formando una teselación de
polígonos de Voronoi.

$$\bar{P} = \frac{\sum_{i=1}^{N} P_i A_i}{A_T}$$

Donde:
- $P_i$: precipitación de la estación $i$.
- $A_i$: área del polígono de influencia de la estación $i$ **dentro de la cuenca**.
- $A_T$: área total de la cuenca.

**Ventaja sobre el promedio aritmético:** Las estaciones más representativas
(con mayor área de influencia) contribuyen más al promedio. Las estaciones fuera
de la cuenca también pueden incluirse si su polígono intercepta el límite.

**Limitación:** Los pesos $A_i/A_T$ son fijos: se calculan una sola vez a partir
de la posición geográfica de las estaciones y no cambian con el evento analizado.



## 8.3 Método de las isoyetas

Las **isoyetas** son líneas de igual precipitación, análogas a las curvas de nivel
topográfico. Se trazan interpolando entre los valores puntuales de cada estación
y permiten calcular un promedio ponderado por el área comprendida entre isoyetas
sucesivas.

$$\bar{P} = \frac{\sum_{i=1}^{N} \bar{P}_{i,i+1} \cdot A_{i,i+1}}{A_T}$$

Donde $\bar{P}_{i,i+1}$ es el valor medio entre dos isoyetas consecutivas y
$A_{i,i+1}$ es el área encerrada entre ellas dentro de la cuenca.

**Ventaja:** Es el método más preciso. Un experto puede incorporar el efecto de
la orografía (relieve) en el trazado, algo que los otros métodos no capturan.

**Limitación:** Las isoyetas son **dinámicas** — deben retrazarse para cada evento
o período analizado, a diferencia de los polígonos de Thiessen que son fijos.
Además, tienen una componente subjetiva en el trazado, aunque esto puede
reducirse con técnicas geoestadísticas como el *kriging*.

## 8.4 Aplicación práctica: cuenca de estudio

Aplicaremos los métodos de estimación espacial usando las cinco estaciones de
la red pluviométrica de la región de Puerto Montt, sobre una cuenca real
delimitada mediante un shapefile de la DGA.

https://camels.cr2.cl/

**Datos de la cuenca:**
- Área: ~250 km²
- Ubicación: Región de Los Lagos, Chile (~41.4°S, 72.9°O)
- Fuente del contorno: shapefile DGA (WGS84)

**Red pluviométrica disponible:**

| Estación | Código | Latitud | Longitud | Fuente |
|----------|--------|---------|----------|--------|
| Cascada | 10410015 | 41.08°S | 72.63°O | DGA |
| Lago Chapo | 10430005 | 41.43°S | 72.58°O | DGA |
| Puerto Montt | 10425001 | 41.46°S | 72.94°O | DGA |
| Tepual | 410005 | 41.45°S | 73.10°O | DMC |
| Puelo | 10523001 | 41.65°S | 72.31°O | DGA |

> **Nota:** En la práctica, los polígonos de Thiessen se calculan con un SIG
> (QGIS, ArcGIS). En este notebook los calculamos directamente en Python usando
> el método de Monte Carlo para estimar las áreas de intersección entre los
> polígonos de Voronoi y el contorno de la cuenca.

In [ ]:
# ── MÉTODO 2: Polígonos de Thiessen ───────────────────────────────────────────
# ── Coordenadas geográficas de las estaciones ─────────────────────────────────
# Formato: (longitud, latitud) en grados decimales.
# Las longitudes al oeste son negativas por convención.
estaciones_geo = {
    'Cascada':     (-72.6328, -41.0781),
    'LagoChapo':   (-72.5764, -41.4297),
    'PuertoMontt': (-72.9378, -41.4600),
    'Tepual':      (-73.0978, -41.4500),
    'Puelo':       (-72.3117, -41.6511),
}

# ── Proyección local: grados geográficos → kilómetros ─────────────────────────
# Para calcular áreas en km² necesitamos trabajar en coordenadas planas.
# Usamos una proyección cilíndrica simple centrada en el centroide de la cuenca.
# A ~41°S: 1° latitud ≈ 111.0 km, 1° longitud ≈ 111.0 × cos(41.4°) ≈ 83.3 km


# Centroide aproximado de la cuenca (calculado del shapefile)
cx_cuenca = -72.9121  # longitud centroide (°)
cy_cuenca = -41.3920  # latitud centroide (°)

R_lat = 111.0                                          # km por grado de latitud
R_lon = 111.0 * math.cos(math.radians(abs(cy_cuenca))) # km por grado de longitud
print(f"Factor de escala longitud: {R_lon:.3f} km/°")
print(f"Factor de escala latitud:  {R_lat:.3f} km/°")

def a_km(lon, lat):
    """Convierte coordenadas geográficas (lon, lat) a coordenadas planas (km)
    relativas al centroide de la cuenca."""
    x = (lon - cx_cuenca) * R_lon
    y = (lat - cy_cuenca) * R_lat
    return (x, y)

# Convertir todas las estaciones a coordenadas planas
estaciones_km = {nombre: a_km(lon, lat) for nombre, (lon, lat) in estaciones_geo.items()}

print("\nPosición de estaciones relativa al centroide de la cuenca (km):")
print(f"{'Estación':<15} {'Este (km)':>10} {'Norte (km)':>11}")
print("-" * 38)
for nombre, (x, y) in estaciones_km.items():
    print(f"{nombre:<15} {x:>10.2f} {y:>11.2f}")

In [ ]:
# ── Leer el contorno de la cuenca desde el shapefile ─────────────────────────
# El shapefile fue descargado de la DGA. Lo leemos directamente con Python
# estándar (sin necesidad de geopandas) parseando el formato binario .shp.


def leer_shapefile_poligono(ruta):
    """Lee el primer polígono de un archivo .shp en formato ESRI Shapefile.
    Retorna una lista de tuplas (lon, lat) con los vértices del polígono."""
    with open(ruta, 'rb') as f:
        f.read(100)  # Omitir cabecera de 100 bytes
        while True:
            hdr = f.read(8)
            if len(hdr) < 8:
                break
            content_len = struct.unpack('>i', hdr[4:8])[0] * 2
            rec = f.read(content_len)
            stype = struct.unpack('<i', rec[:4])[0]
            if stype == 5:  # Tipo 5 = Polygon
                num_parts  = struct.unpack('<i', rec[36:40])[0]
                num_points = struct.unpack('<i', rec[40:44])[0]
                base = 44 + 4 * num_parts
                puntos = []
                for k in range(num_points):
                    x, y = struct.unpack('<dd', rec[base + 16*k : base + 16*k + 16])
                    puntos.append((x, y))
                return puntos

# Cargar el shapefile de la cuenca
# Ajusta la ruta si ejecutas el notebook localmente
ruta_shp = "Data/polygon_10411002_RioNegroLasLomas/polygon.shp"  # ← modifica esta ruta si es necesario
cuenca_geo = leer_shapefile_poligono(ruta_shp)

# Proyectar la cuenca a coordenadas planas (km)
cuenca_km = [a_km(p[0], p[1]) for p in cuenca_geo]

xs_k = [p[0] for p in cuenca_km]
ys_k = [p[1] for p in cuenca_km]

print(f"Cuenca cargada: {len(cuenca_geo)} vértices")
print(f"Extensión en km desde el centroide:")
print(f"  Este–Oeste: {min(xs_k):.1f} → {max(xs_k):.1f} km")
print(f"  Sur–Norte:  {min(ys_k):.1f} → {max(ys_k):.1f} km")


# ── Funciones auxiliares para el método de Thiessen ──────────────────────────

def punto_en_poligono(px, py, poligono):
    """
    Determina si el punto (px, py) está dentro de 'poligono'
    usando el algoritmo de ray casting.
    Un rayo horizontal desde el punto hacia la derecha: si cruza un número
    impar de aristas del polígono, el punto es interior.
    """
    n = len(poligono)
    dentro = False
    j = n - 1
    for i in range(n):
        xi, yi = poligono[i]
        xj, yj = poligono[j]
        if ((yi > py) != (yj > py)) and \
           (px < (xj - xi) * (py - yi) / (yj - yi + 1e-15) + xi):
            dentro = not dentro
        j = i
    return dentro

def estacion_mas_cercana(px, py, sitios):
    """
    Retorna el nombre de la estación más cercana al punto (px, py).
    Esta es la base del diagrama de Voronoi: cada punto del espacio
    pertenece a la celda de la estación más cercana.
    """
    min_d = float('inf')
    mejor = None
    for nombre, (sx, sy) in sitios.items():
        dist = (px - sx)**2 + (py - sy)**2  # distancia euclidiana al cuadrado
        if dist < min_d:
            min_d = dist
            mejor = nombre
    return mejor

print("\n✅ Funciones auxiliares definidas.")

In [ ]:
# ── Cálculo de áreas de Voronoi dentro de la cuenca (Monte Carlo) ─────────────
#
# Estrategia:
# 1. Generamos N puntos aleatorios dentro del bounding box de la cuenca.
# 2. Para cada punto dentro de la cuenca, determinamos a qué estación pertenece
#    (la más cercana = asignación de Voronoi).
# 3. La fracción de puntos asignados a cada estación estima el área de su
#    polígono de Thiessen dentro de la cuenca.
#
# El error del método de Monte Carlo es ~1/√N. Con N=80.000 el error es ~0.35%.



xmin_k, xmax_k = min(xs_k), max(xs_k)
ymin_k, ymax_k = min(ys_k), max(ys_k)
area_bbox = (xmax_k - xmin_k) * (ymax_k - ymin_k)  # km²

N = 80_000
random.seed(42)  # Semilla fija para reproducibilidad

conteos = {nombre: 0 for nombre in estaciones_km}
total_dentro = 0

for _ in range(N):
    px = random.uniform(xmin_k, xmax_k)
    py = random.uniform(ymin_k, ymax_k)
    if punto_en_poligono(px, py, cuenca_km):
        total_dentro += 1
        estacion = estacion_mas_cercana(px, py, estaciones_km)
        conteos[estacion] += 1

# Área total de la cuenca estimada por Monte Carlo
area_cuenca_km2 = area_bbox * total_dentro / N

# Área de cada polígono de Thiessen dentro de la cuenca
areas_thiessen = {n: area_bbox * c / N for n, c in conteos.items()}

# Pesos de Thiessen: Ai / AT
pesos_thiessen = {n: conteos[n] / total_dentro for n in estaciones_km}

# Resumen
print(f"Área cuenca estimada (Monte Carlo, N={N:,}): {area_cuenca_km2:.1f} km²\n")
print(f"{'Estación':<15} {'Área Ai (km²)':>14} {'Peso Ai/AT':>12} {'%':>8}")
print("─" * 52)
for nombre in estaciones_km:
    a = areas_thiessen[nombre]
    w = pesos_thiessen[nombre]
    barra = "█" * int(w * 40)
    print(f"{nombre:<15} {a:>14.1f} {w:>12.4f} {w*100:>7.1f}%  {barra}")
print("─" * 52)
print(f"{'TOTAL':<15} {sum(areas_thiessen.values()):>14.1f} {sum(pesos_thiessen.values()):>12.4f}")

In [ ]:
# ── Mapa de polígonos de Thiessen sobre la cuenca ────────────────────────────
#
# Visualizamos el diagrama de Voronoi coloreado por estación, recortado al
# contorno de la cuenca. Cada color representa el área de influencia de
# una estación pluviométrica.



colores_est = {
    'PuertoMontt': '#4C72B0',
    'Tepual':      '#DD8452',
    'LagoChapo':   '#55A868',
    'Cascada':     '#C44E52',
    'Puelo':       '#8172B3',
}

# Generar grilla de puntos para colorear el mapa
GRID = 120
grid_x = np.linspace(xmin_k, xmax_k, GRID)
grid_y = np.linspace(ymin_k, ymax_k, GRID)

fig, ax = plt.subplots(figsize=(9, 8))

# Colorear la cuenca por estación más cercana (diagrama de Voronoi)
for gx in grid_x:
    for gy in grid_y:
        if punto_en_poligono(gx, gy, cuenca_km):
            estacion = estacion_mas_cercana(gx, gy, estaciones_km)
            ax.plot(gx, gy, 's', color=colores_est[estacion],
                    markersize=4.5, alpha=0.6, markeredgewidth=0)

# Contorno de la cuenca
xs_plot = [p[0] for p in cuenca_km[::5]] + [cuenca_km[0][0]]
ys_plot = [p[1] for p in cuenca_km[::5]] + [cuenca_km[0][1]]
ax.plot(xs_plot, ys_plot, 'k-', linewidth=1.8, label='Límite cuenca')

# Estaciones pluviométricas
for nombre, (x, y) in estaciones_km.items():
    w = pesos_thiessen[nombre]
    ax.plot(x, y, 'o', color=colores_est[nombre],
            markersize=10, markeredgecolor='black', markeredgewidth=1.2, zorder=5)
    ax.annotate(
        f"{nombre}\n({w*100:.1f}%)",
        (x, y), textcoords="offset points", xytext=(8, 5),
        fontsize=8.5, fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.3', fc='white', alpha=0.8, ec='gray')
    )

# Leyenda de colores
leyenda = [mpatches.Patch(color=colores_est[n], alpha=0.7,
           label=f"{n}  ({areas_thiessen[n]:.1f} km²,  {pesos_thiessen[n]*100:.1f}%)")
           for n in estaciones_km if pesos_thiessen[n] > 0]
ax.legend(handles=leyenda, loc='lower right', fontsize=9, title="Área de influencia")

ax.set_xlabel("Distancia Este–Oeste desde centroide (km)", fontsize=11)
ax.set_ylabel("Distancia Sur–Norte desde centroide (km)", fontsize=11)
ax.set_title("Polígonos de Thiessen – Cuenca de estudio\n"
             f"Área total: {area_cuenca_km2:.0f} km²", fontsize=13)
ax.set_aspect('equal')
ax.grid(linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# ── Calcular precipitación media espacial con ambos métodos ──────────────────

# Precipitación anual por estación (reutilizamos df_anual de secciones previas)
# Incluimos la serie original de Puerto Montt (sin datos faltantes artificiales)
df_espacial = pd.concat(
    {
        'PuertoMontt': PMontt_original['precipitacion_diaria_mm'],
        'LagoChapo':   LagoChapo['precipitacion_diaria_mm'],
        'Puelo':       Puelo['precipitacion_diaria_mm'],
        'Tepual':      Tepual['precipitacion_diaria_mm'],
        'Cascada':     Cascada['precipitacion_diaria_mm'],
    },
    axis=1
)

# Precipitación anual total (años con al menos 300 días de dato)
df_anual_esp = df_espacial.resample('YE').sum(min_count=300)
df_anual_esp.index = df_anual_esp.index.year

# ── MÉTODO 1: Promedio aritmético simple ──────────────────────────────────────
# P_barra = (P1 + P2 + ... + PN) / N  (solo estaciones con dato ese año)
df_anual_esp['Aritmetico'] = df_anual_esp[list(estaciones_km.keys())].mean(axis=1)

# ── MÉTODO 2: Polígonos de Thiessen ───────────────────────────────────────────
# P_barra = Σ (Pi × Ai/AT)
# Los pesos calculados por Monte Carlo son los factores reales de ponderación.
# Estaciones con peso 0 (Cascada, Puelo) no contribuyen: su polígono de
# Voronoi no intercepta la cuenca en estudio.
df_anual_esp['Thiessen'] = (
      df_anual_esp['PuertoMontt'] * pesos_thiessen['PuertoMontt']
    + df_anual_esp['LagoChapo']   * pesos_thiessen['LagoChapo']
    + df_anual_esp['Tepual']      * pesos_thiessen['Tepual']
    + df_anual_esp['Cascada']     * pesos_thiessen['Cascada']
    + df_anual_esp['Puelo']       * pesos_thiessen['Puelo']
)

# ── Tabla comparativa ─────────────────────────────────────────────────────────
diferencia = df_anual_esp['Thiessen'] - df_anual_esp['Aritmetico']
df_resumen = pd.DataFrame({
    'Aritmético (mm)': df_anual_esp['Aritmetico'].round(1),
    'Thiessen (mm)':   df_anual_esp['Thiessen'].round(1),
    'Diferencia (mm)': diferencia.round(1),
    'Dif. (%)':        (diferencia / df_anual_esp['Aritmetico'] * 100).round(1),
})

print("Comparación anual de métodos de estimación espacial:")
print(df_resumen.to_string())

print(f"\n{'Módulo pluviométrico espacial (media histórica)':}")
print(f"  Promedio aritmético: {df_anual_esp['Aritmetico'].mean():.1f} mm/año")
print(f"  Thiessen:            {df_anual_esp['Thiessen'].mean():.1f} mm/año")

## 8.5 Interpretación de los resultados

Los pesos de Thiessen calculados para esta cuenca reflejan su geometría real:

| Estación | Área de influencia | Peso $A_i/A_T$ | Observación |
|----------|--------------------|----------------|-------------|
| Puerto Montt | ~202 km² | 80.8% | Única estación dentro de la cuenca |
| Tepual | ~44 km² | 17.6% | Sector oeste de la cuenca |
| Lago Chapo | ~4 km² | 1.6% | Pequeña franja este |
| Cascada | 0 km² | 0% | Su polígono no alcanza la cuenca |
| Puelo | 0 km² | 0% | Su polígono no alcanza la cuenca |

**Diferencia entre métodos:**

- El promedio aritmético asigna peso igual (20%) a las 5 estaciones, incluyendo
  Cascada y Puelo, que están muy alejadas de la cuenca.
- Thiessen asigna el peso según la representatividad geográfica real de cada estación.
- Cuando existe una sola estación claramente dominante dentro de la cuenca, la
  diferencia entre métodos puede ser significativa.



---
# 9. Intensidad de precipitación

## Concepto

La **intensidad de precipitación** ($i$) es la cantidad de lluvia que cae por unidad de tiempo. Se expresa habitualmente en **mm/h** o mm/min.

$$i = \frac{\Delta P}{\Delta t}$$

Donde:
- $\Delta P$: precipitación registrada en el intervalo (mm).
- $\Delta t$: duración del intervalo de tiempo (h o min).

## ¿Por qué es importante la intensidad?

La intensidad es una variable crítica en ingeniería hidráulica e hidrología:

- **Diseño de alcantarillados:** Determina el caudal de diseño mediante el método racional   ($Q = C \cdot i \cdot A$).
- **Erosión del suelo:** Lluvias de alta intensidad generan escorrentía superficial y erosión   aun cuando la precipitación total no sea elevada.
- **Infiltración:** Si $i$ supera la tasa de infiltración del suelo, se genera escorrentía   incluso en suelos con capacidad de retención alta.
- **Alertas tempranas:** Las alertas de crecidas se basan frecuentemente en umbrales de intensidad.

## Curvas IDF (Intensidad – Duración – Frecuencia)

Para el diseño hidrológico se utilizan las **curvas IDF**, que relacionan la intensidad de diseño con la duración del evento y su período de retorno. Su construcción requiere datos de pluviógrafos (registro continuo) y análisis de frecuencia estadística.
